# 03 — Analytical modelling (Gold)

A category manager asks two different questions of the same retailers:

1. **What did shoppers pay?** For which product, at which store, in which week,
   and was it on promotion?
2. **What is in the product?** Energy, fats, sugars and salt, per 100 g or 100 ml.

Gold answers each question with its own fact table and shares the same product,
retailer and calendar dimensions. Every builder below is the production code in
[`gold.py`](../src/retail_pricing/gold.py); this notebook calls it step by step and
shows one piece of evidence after each call. Nothing is re-implemented here, and
nothing is published: [`pipeline.py`](../src/retail_pricing/pipeline.py) writes
`data/gold/`.

| Process | Fact grain | Dimensions |
| --- | --- | --- |
| Weekly pricing and promotion | product × retailer location × week | Product, Shop, Location, Date |
| Product nutrition | product × observation date × nutrient × basis | Product, Shop, Nutrient, Date |

Categories reach the facts through `Product` and a bridge; a product can sit in
several retailer categories, so no fact carries a category key.

**Belgian client mart.** Silver keeps every market (BE, NL, LU, DE). Gold publishes the
Belgian mart: weekly prices are already Belgian, nutrition keeps only products whose
country is Belgium, and every dimension holds only members that a fact refers to. The
calendar stays whole so that time navigation has no holes.

See the [Gold model notes](../docs/gold-model.md) for the decisions and limits.

## 1. Setup

Load the Silver checkpoint through its manifest. A missing, stale or edited Silver
stops the notebook instead of modelling the wrong data.

In [1]:
import duckdb
import polars as pl
from IPython.display import Markdown, display

from retail_pricing import SILVER_DIR, SILVER_FILES
from retail_pricing.gold import (
    add_price_comparison_columns,
    attach_primary_categories,
    build_category_model,
    build_dim_date,
    build_dim_location,
    build_dim_nutrient,
    build_dim_product,
    build_dim_shop,
    create_fact_product_nutrition,
    create_fact_weekly_prices,
)
from retail_pricing.gold_validation import (
    validate_gold_model,
    validate_mart_reconciliation,
)
from retail_pricing.pipeline import load_silver_manifest

pl.Config.set_tbl_rows(12)
pl.Config.set_fmt_str_lengths(60)
silver = {name: SILVER_DIR / filename for name, filename in SILVER_FILES.items()}
manifest = load_silver_manifest(SILVER_DIR)
pl.DataFrame([{"dataset": f["dataset"], "rows": f["rows"]} for f in manifest["files"]])

dataset,rows
str,i64
"""products""",32826
"""locations""",1638
"""prices""",1198547
"""nutritionals""",9467978
"""weekly_products""",114517
"""weekly_locations""",1230
"""weekly_prices""",19075099
"""product_category_paths""",154760
"""product_primary_categories""",167077


## 2. Dimensions

### 2.1 Date

One calendar shared by both facts. It covers whole calendar years so that time
navigation is complete; the facts keep their own, shorter history. Weeks run
Monday to Sunday and ISO week numbers keep the year boundary honest.

In [2]:
dim_date = build_dim_date(silver["weekly_prices"], silver["nutritionals"])
dim_date.filter(
    pl.col("date").is_between(pl.date(2020, 12, 28), pl.date(2021, 1, 6))
).select(
    "date", "iso_year", "iso_week", "year_week", "week_start_date", "week_end_date"
)

date,iso_year,iso_week,year_week,week_start_date,week_end_date
date,i32,i32,str,date,date
2020-12-28,2020,53,"""2020-W53""",2020-12-28,2021-01-03
2020-12-29,2020,53,"""2020-W53""",2020-12-28,2021-01-03
2020-12-30,2020,53,"""2020-W53""",2020-12-28,2021-01-03
2020-12-31,2020,53,"""2020-W53""",2020-12-28,2021-01-03
2021-01-01,2020,53,"""2020-W53""",2020-12-28,2021-01-03
2021-01-02,2020,53,"""2020-W53""",2020-12-28,2021-01-03
2021-01-03,2020,53,"""2020-W53""",2020-12-28,2021-01-03
2021-01-04,2021,1,"""2021-W01""",2021-01-04,2021-01-10
2021-01-05,2021,1,"""2021-W01""",2021-01-04,2021-01-10


In [3]:
display(
    Markdown(
        f"**Date:** {dim_date.height:,} days, {dim_date['date'].min()} to "
        f"{dim_date['date'].max()}, one row and one key per day. ISO week 53 of 2020 "
        "runs into January 2021, as shown above."
    )
)

**Date:** 1,096 days, 2019-01-01 to 2021-12-31, one row and one key per day. ISO week 53 of 2020 runs into January 2021, as shown above.

### 2.2 Shop

One row per retailer code. The reporting name comes from Silver, where the reviewed
code-to-name reference lives, so Gold never guesses a retailer.

In [4]:
dim_shop = build_dim_shop(silver["weekly_prices"], silver["nutritionals"])
duck = duckdb.connect()
pricing = duck.sql(
    f"SELECT DISTINCT shop FROM read_parquet('{silver['weekly_prices'].as_posix()}')"
).pl()["shop"]
nutrition = duck.sql(
    f"SELECT DISTINCT shop FROM read_parquet('{silver['nutritionals'].as_posix()}') "
    "WHERE country = 'be'"
).pl()["shop"]
dim_shop.with_columns(
    pl.col("shop").is_in(pricing.to_list()).alias("has_prices"),
    pl.col("shop").is_in(nutrition.to_list()).alias("has_nutrition"),
)

shop_key,shop,shop_name,has_prices,has_nutrition
i64,str,str,bool,bool
1,"""ah""","""Albert Heijn""",false,true
2,"""aldi""","""Aldi""",true,false
3,"""cg""","""Collect&Go""",false,true
4,"""clp""","""Colruyt Lowest Prices""",true,true
5,"""crf""","""Carrefour""",true,true
6,"""dll""","""Delhaize""",true,true


In [5]:
display(
    Markdown(
        f"**Shop:** {dim_shop.height} retailers in the Belgian mart. Prices cover "
        f"{len(pricing)} of them and Belgian nutrition {len(nutrition)}; a retailer needs to "
        "appear in only one process. Jumbo has only Dutch nutrition, so it is not in the mart."
    )
)

**Shop:** 6 retailers in the Belgian mart. Prices cover 4 of them and Belgian nutrition 5; a retailer needs to appear in only one process. Jumbo has only Dutch nutrition, so it is not in the mart.

### 2.3 Product

A product is identified by `(daltix_id, shop, country)`; the same id in another
retailer or country is a different product. Descriptions come from the weekly
product reference and are filled, field by field, from the products reference.

Weekly prices carry only `(daltix_id, shop)`. An unmatched price identity is
recovered when the products reference supports exactly one country. Every other one
gets its **own inferred member** named "Unmatched product (shop id)", with country
`be` from the Belgian pricing process, no invented attributes and the FMCG status
`missing_evidence`. Prices therefore never share one Unknown row; `product_key = -1`
stays only as the reserved member for a price with no id at all, which does not occur.
The mart keeps only products that a fact refers to: priced products and products with
Belgian nutrition. The English FMCG category is attached from Silver, where it is
decided once per product.

In [6]:
dim_product = build_dim_product(
    silver["weekly_products"],
    silver["nutritionals"],
    silver["products"],
    silver["weekly_prices"],
)
dim_product = attach_primary_categories(
    dim_product, silver["product_primary_categories"]
)
dim_product.group_by("descriptive_source").len().sort("len", descending=True)

descriptive_source,len
str,u32
"""weekly_products""",89110
"""inferred_from_prices""",11011
"""identity_only""",9604
"""products""",3452
"""unknown""",1


In [7]:
dim_product.group_by("fmcg_classification_status").len().sort("len", descending=True)

fmcg_classification_status,len
str,u32
"""classified""",70159
"""missing_evidence""",25158
"""no_supported_match""",10273
"""excluded_evidence""",6092
"""conflicting_evidence""",1456
"""review_required""",39
"""unknown_product""",1


In [8]:
classified = dim_product.filter(pl.col("fmcg_classification_status") == "classified")
inferred_members = dim_product.filter(pl.col("is_inferred_product")).height
display(
    Markdown(
        f"**Product:** {dim_product.height:,} members: {dim_product.height - 1 - inferred_members:,} "
        f"described by a reference, {inferred_members:,} inferred from prices and the "
        f"reserved Unknown member. {classified.height:,} "
        f"({classified.height / dim_product.height:.1%}) have a rule-supported FMCG category; "
        "the rest keep an explicit reason (missing, conflicting or unsupported category "
        "evidence). Coverage is not accuracy."
    )
)

**Product:** 113,178 members: 102,166 described by a reference, 11,011 inferred from prices and the reserved Unknown member. 70,159 (62.0%) have a rule-supported FMCG category; the rest keep an explicit reason (missing, conflicting or unsupported category evidence). Coverage is not accuracy.

### 2.4 Category model

Retailer category paths are kept as evidence. Gold uses only paths that are not
duplicates, not ancestors of another path, not site navigation and not repeated
labels. A bridge keeps a product's several categories without multiplying either
fact. These are retailer classifications, not a shared taxonomy; the FMCG category
above is the one to compare across retailers.

In [9]:
dim_category, bridge_product_category = build_category_model(
    silver["product_category_paths"], dim_product
)
duck.register("dim_product", dim_product)
duck.register("dim_category", dim_category)
duck.register("bridge", bridge_product_category)
duck.sql(
    """
    WITH memberships AS (
        SELECT product_key, COUNT(*) AS n FROM bridge GROUP BY product_key
    )
    SELECT COUNT(*) FILTER (WHERE m.n IS NULL) AS products_without_category,
           COUNT(*) FILTER (WHERE m.n = 1) AS with_one_category,
           COUNT(*) FILTER (WHERE m.n > 1) AS with_several_categories,
           MAX(m.n) AS most_categories_for_one_product
    FROM dim_product p LEFT JOIN memberships m USING (product_key)
    """
).pl()

products_without_category,with_one_category,with_several_categories,most_categories_for_one_product
i64,i64,i64,i64
29184,80467,3527,8


In [10]:
overlap = duck.sql(
    """
    SELECT COUNT(*) AS shorter_and_longer_path_pairs,
           COUNT(DISTINCT a.product_key) AS affected_products
    FROM bridge a
    JOIN dim_category ca ON a.category_key = ca.category_key
    JOIN bridge b ON a.product_key = b.product_key
    JOIN dim_category cb ON b.category_key = cb.category_key
    WHERE len(ca.category_path_normalized) < len(cb.category_path_normalized)
      AND ca.category_path_normalized
          = list_slice(cb.category_path_normalized, 1, len(ca.category_path_normalized))
    """
).pl()
display(
    Markdown(
        f"**Categories:** {dim_category.height:,} retailer categories and "
        f"{bridge_product_category.height:,} product memberships. "
        f"{overlap['affected_products'][0]} products keep both a shorter and a longer "
        "path from different sources; no reviewed rule says which is obsolete, so both stay. "
        "Category totals overlap and must not be added up."
    )
)
overlap

**Categories:** 6,502 retailer categories and 88,493 product memberships. 26 products keep both a shorter and a longer path from different sources; no reviewed rule says which is obsolete, so both stay. Category totals overlap and must not be added up.

shorter_and_longer_path_pairs,affected_products
i64,i64
26,26


### 2.5 Location

One row per `(shop, location)` that has weekly prices. Silver adds two deterministic
attributes: `location_type` (a `Default` location is a national base price context,
every other one a store) and `region` (Brussels, Flanders or Wallonia from a
four-digit Belgian postcode, empty when there is no such postcode). A far-away
coordinate is flagged, never moved. The map should show the stores only; the base
price contexts have no coordinates.

In [11]:
dim_location = build_dim_location(silver["weekly_locations"], silver["weekly_prices"])
dim_location.select(
    "location_key", "shop", "location", "location_type", "region", "postcode", "state"
)

location_key,shop,location,location_type,region,postcode,state
i64,str,str,str,str,str,str
1,"""aldi""","""base""","""National price (online)""",null,null,null
2,"""clp""","""anderlecht-veeweyde""","""Store""",null,null,"""Région de Bruxelles-Capitale - Brussels Hoofdstedelijk Gewes…"
3,"""clp""","""ans""","""Store""","""Wallonia""","""4430""","""Wallonie"""
4,"""clp""","""beveren""","""Store""",null,null,null
5,"""clp""","""halle""","""Store""","""Flanders""","""1500""","""Vlaanderen"""
6,"""crf""","""base""","""National price (online)""",null,null,null
…,…,…,…,…,…,…
11,"""crf""","""mont-st-jean""","""Store""","""Wallonia""","""1410""","""Wallonie"""
12,"""crf""","""sint-agatha-berchem""","""Store""","""Brussels""","""1083""","""Région de Bruxelles-Capitale - Brussels Hoofdstedelijk Gewes…"


In [12]:
stores = dim_location.filter(pl.col("location_type") == "Store")
without_region = stores.filter(pl.col("region").is_null())
display(
    Markdown(
        f"**Location:** {dim_location.height:,} price contexts: "
        f"{dim_location.height - stores.height} national base prices and {stores.height} stores. "
        f"{without_region.height} stores have no four-digit postcode and therefore no region "
        f"({', '.join(without_region['location'])}). The price data covers a handful of "
        "stores, far fewer than the retailer networks; do not read a price as national."
    )
)

**Location:** 16 price contexts: 3 national base prices and 13 stores. 2 stores have no four-digit postcode and therefore no region (anderlecht-veeweyde, beveren). The price data covers a handful of stores, far fewer than the retailer networks; do not read a price as national.

### 2.6 Nutrient

One row per canonical nutrient, with the single unit Silver standardised it to
(`g`, `kJ` or `kcal`). Energy in kJ and in kcal remain separate nutrients.

In [13]:
dim_nutrient = build_dim_nutrient(silver["nutritionals"])
dim_nutrient

nutrient_key,nutrient_name,standard_unit
i64,str,str
1,"""carbohydrates""","""g"""
2,"""energy""","""kJ"""
3,"""fats""","""g"""
4,"""fibers""","""g"""
5,"""kilocalories""","""kcal"""
6,"""monounsaturated_fats""","""g"""
…,…,…
9,"""proteins""","""g"""
10,"""salt""","""g"""


## 3. Facts

### 3.1 Weekly prices

Silver keeps every distinct price seen for a product, store and week. Gold turns
them into one row per week without picking a "true" price: a single regular,
promotional or effective price is filled in **only when all observations agree**,
and the range, the number of versions and variation flags always stay.

In [14]:
con = duckdb.connect()
create_fact_weekly_prices(
    con, silver["weekly_prices"], dim_product, dim_shop, dim_location, dim_date
)
add_price_comparison_columns(con)
con.sql(
    """
    SELECT COUNT(*) AS weekly_rows,
           COUNT(*) FILTER (WHERE has_intraweek_price_variation) AS with_price_variation,
           COUNT(*) FILTER (WHERE effective_price IS NULL) AS without_single_effective_price,
           COUNT(*) FILTER (WHERE promotion_state_conflict) AS uncertain_promotion,
           COUNT(*) FILTER (WHERE discount_pct IS NOT NULL) AS with_discount
    FROM fact_weekly_prices
    """
).pl()

weekly_rows,with_price_variation,without_single_effective_price,uncertain_promotion,with_discount
i64,i64,i64,i64,i64
18699187,375912,371087,35704,1554500


In [15]:
unmatched = con.sql(
    """
    SELECT COUNT(*) AS rows,
           COUNT(*) FILTER (WHERE f.product_key = -1) AS unknown_rows,
           COUNT(*) FILTER (WHERE p.is_inferred_product) AS inferred_rows,
           COUNT(DISTINCT f.product_key) FILTER (WHERE p.is_inferred_product)
               AS inferred_members
    FROM fact_weekly_prices f JOIN dim_product p USING (product_key)
    """
).pl()
rows, unknown_rows, inferred_rows, inferred_members = unmatched.row(0)
silver_rows = next(
    f["rows"] for f in manifest["files"] if f["dataset"] == "weekly_prices"
)
display(
    Markdown(
        f"**Weekly prices:** {silver_rows:,} Silver observations become {rows:,} weekly "
        f"rows, one per product, store and week. {inferred_rows:,} rows "
        f"({inferred_rows / rows:.2%}) belong to {inferred_members:,} price identities that no "
        "product reference describes. Each has its own inferred product member, so their "
        f"prices count in price analysis without being merged; {unknown_rows:,} rows use the "
        "reserved Unknown member."
    )
)

**Weekly prices:** 19,075,099 Silver observations become 18,699,187 weekly rows, one per product, store and week. 476,695 rows (2.55%) belong to 11,011 price identities that no product reference describes. Each has its own inferred product member, so their prices count in price analysis without being merged; 0 rows use the reserved Unknown member.

#### Like-for-like price ratios

A category manager wants to know whether shoppers paid more than a year ago and
whether that comes from higher shelf prices or from fewer promotions. The weekly
prices therefore carry three ratio columns **on the same rows** (same grain, no extra
fact):

- `log_paid_yoy_ratio`: the effective (paid) price against the same product and
  location 364 days earlier;
- `log_shelf_yoy_ratio`: the same with the regular (shelf) price, so the gap between
  the two is the effect of promotions;
- `log_store_vs_national_ratio`: a store price against its retailer's national online
  price for the same product and week.

They are natural logs because the geometric mean of price relatives,
`exp(mean(log ratio)) - 1`, can then be taken over any slice of weeks, retailers or
categories. Every weekly source price carries a random ±10% perturbation (factor
0.9, 1.0 or 1.1), which only cancels out over many pairs; the semantic model
therefore shows a slice only from 500 pairs.

In [16]:
ratios = con.sql(
    """
    SELECT COUNT(*) AS weekly_rows,
           COUNT(log_paid_yoy_ratio) AS paid_yoy_pairs,
           COUNT(log_shelf_yoy_ratio) AS shelf_yoy_pairs,
           COUNT(log_store_vs_national_ratio) AS store_vs_national_pairs,
           EXP(AVG(log_paid_yoy_ratio)) - 1 AS paid_yoy,
           EXP(AVG(log_shelf_yoy_ratio)) - 1 AS shelf_yoy
    FROM fact_weekly_prices
    """
).pl()
(
    weekly_rows,
    paid_pairs,
    shelf_pairs,
    store_pairs,
    paid_yoy,
    shelf_yoy,
) = ratios.row(0)
display(
    Markdown(
        f"**Price ratios:** the {weekly_rows:,} weekly rows are unchanged. "
        f"{paid_pairs:,} carry a paid and {shelf_pairs:,} a shelf year-on-year ratio, "
        f"and {store_pairs:,} store rows a ratio against the national price. Across the "
        f"whole mart, like-for-like prices paid rose {paid_yoy:+.2%} and shelf prices "
        f"{shelf_yoy:+.2%}: the gap between them is what promotions absorbed."
    )
)
ratios

**Price ratios:** the 18,699,187 weekly rows are unchanged. 5,645,473 carry a paid and 5,647,663 a shelf year-on-year ratio, and 9,284,071 store rows a ratio against the national price. Across the whole mart, like-for-like prices paid rose +1.66% and shelf prices +1.20%: the gap between them is what promotions absorbed.

weekly_rows,paid_yoy_pairs,shelf_yoy_pairs,store_vs_national_pairs,paid_yoy,shelf_yoy
i64,i64,i64,i64,f64,f64
18699187,5645473,5647663,9284071,0.016625,0.012015


### 3.2 Product nutrition

One row per product, observation date, nutrient and basis. Per-100 g and per-100 ml
values are never mixed, and the download date is an observation date rather than a
promise that the label was valid earlier. Values with an incompatible unit keep
their row and lose only the canonical value.

In [17]:
create_fact_product_nutrition(
    con, silver["nutritionals"], dim_product, dim_shop, dim_date, dim_nutrient
)
con.sql(
    """
    SELECT nutrition_basis,
           COUNT(*) AS rows,
           COUNT(*) FILTER (WHERE is_value_usable) AS usable_values,
           COUNT(*) FILTER (WHERE dq_incompatible_nutrient_unit) AS incompatible_unit,
           COUNT(*) FILTER (WHERE dq_mass_exceeds_portion) AS mass_above_portion
    FROM fact_product_nutrition GROUP BY nutrition_basis ORDER BY nutrition_basis
    """
).pl()

nutrition_basis,rows,usable_values,incompatible_unit,mass_above_portion
str,i64,i64,i64,i64
"""per_100g""",3874655,3874617,38,48
"""per_100ml""",459404,459375,29,0


In [18]:
nutrition_rows = con.sql("SELECT COUNT(*) FROM fact_product_nutrition").fetchone()[0]
silver_nutrition = next(
    f["rows"] for f in manifest["files"] if f["dataset"] == "nutritionals"
)
display(
    Markdown(
        f"**Nutrition:** {nutrition_rows:,} of {silver_nutrition:,} Silver nutrient rows "
        "reach the fact; none are dropped by a join. `is_value_usable` means a canonical "
        "value exists, not that the label is biologically plausible."
    )
)

**Nutrition:** 4,334,059 of 9,467,978 Silver nutrient rows reach the fact; none are dropped by a join. `is_value_usable` means a canonical value exists, not that the label is biologically plausible.

#### Latest observation

A label is scraped again and again, so a plain average counts a product once per
download and lets frequently scraped products dominate a benchmark. Gold therefore
flags `is_latest_observation`: the most recent date of every product, nutrient and
basis. Category benchmarks in Power BI use only those rows, so every product weighs
the same.

In [19]:
con.register("dim_nutrient", dim_nutrient)
con.sql(
    """
    SELECT n.nutrient_name,
           COUNT(*) AS observations,
           COUNT(*) FILTER (WHERE f.is_latest_observation) AS latest_only,
           ROUND(AVG(f.nutrient_value), 3) AS average_all_observations,
           ROUND(AVG(f.nutrient_value) FILTER (WHERE f.is_latest_observation), 3)
               AS average_latest_only
    FROM fact_product_nutrition f JOIN dim_nutrient n USING (nutrient_key)
    WHERE f.is_value_usable AND f.nutrition_basis = 'per_100g'
    GROUP BY n.nutrient_name ORDER BY observations DESC LIMIT 6
    """
).pl()

nutrient_name,observations,latest_only,average_all_observations,average_latest_only
str,i64,i64,f64,f64
"""proteins""",444731,24084,8.768,8.663
"""carbohydrates""",443832,24053,28.235,28.909
"""salt""",441255,23933,1.221,1.173
"""fats""",440268,23937,14.724,14.935
"""sugars""",440026,23885,12.901,13.561
"""kilocalories""",439129,23862,280.35,284.62


In [20]:
labels = con.sql(
    """
    SELECT COUNT(*) AS rows,
           COUNT(*) FILTER (WHERE is_latest_observation) AS labels
    FROM fact_product_nutrition
    """
).pl()
rows, label_count = labels.row(0)
display(
    Markdown(
        f"**Latest observation:** {label_count:,} product-nutrient-basis labels carry "
        f"{rows:,} observations, {rows / label_count:.1f} dates each on average. The "
        "average of a nutrient changes when a product is counted once, which is why "
        "benchmarks use the flag."
    )
)

**Latest observation:** 243,673 product-nutrient-basis labels carry 4,334,059 observations, 17.8 dates each on average. The average of a nutrient changes when a product is counted once, which is why benchmarks use the flag.

### 3.3 Belgian scope

What the mart leaves out is measured, not assumed. Silver keeps every market, and
every Belgian Silver row must reach its fact table.

In [21]:
nutrition_by_market = con.sql(
    f"""
    SELECT country, COUNT(*) AS silver_rows
    FROM read_parquet('{silver["nutritionals"].as_posix()}')
    GROUP BY country ORDER BY country
    """
).pl()
validate_mart_reconciliation(
    con, silver["weekly_prices"].as_posix(), silver["nutritionals"].as_posix()
)
nutrition_by_market.with_columns(
    pl.when(pl.col("country") == "be")
    .then(pl.lit("kept in fact_product_nutrition"))
    .otherwise(pl.lit("outside the Belgian mart"))
    .alias("in_gold")
)

country,silver_rows,in_gold
str,i64,str
"""be""",4334059,"""kept in fact_product_nutrition"""
"""nl""",5133919,"""outside the Belgian mart"""


## 4. Validation

The production contract runs on exactly these objects and raises on the first violation. It checks structure that must hold for any snapshot: unique surrogate and natural keys, category-bridge integrity, unique fact grains, foreign keys, and a mart with no unused member (one market, inferred products flagged, a consistent latest-observation flag).

It deliberately holds **no snapshot-specific number**. Row counts and reviewed metrics live in `parity_baseline.json` and are checked by `pipeline parity`, so a new Raw snapshot needs a reviewed baseline rather than a code change.

In [22]:
validate_gold_model(
    con,
    dim_date=dim_date,
    dim_shop=dim_shop,
    dim_product=dim_product,
    dim_category=dim_category,
    bridge_product_category=bridge_product_category,
    dim_location=dim_location,
    dim_nutrient=dim_nutrient,
)
print("Gold contract passed.")

Gold contract passed.


## 5. What the model can and cannot support

### 5.1 Coverage of categories in each fact

Joining a fact to the category bridge repeats rows for products in several
categories, so a category view is a slice, not an additive breakdown.

In [23]:
con.register("bridge", bridge_product_category)
con.sql(
    """
    WITH in_bridge AS (SELECT DISTINCT product_key FROM bridge)
    SELECT 'weekly prices' AS fact, COUNT(*) AS rows,
           COUNT(*) FILTER (WHERE b.product_key IS NOT NULL) AS rows_with_category,
           ROUND(100.0 * COUNT(*) FILTER (WHERE b.product_key IS NOT NULL) / COUNT(*), 2)
               AS coverage_pct
    FROM fact_weekly_prices f LEFT JOIN in_bridge b USING (product_key)
    UNION ALL
    SELECT 'nutrition', COUNT(*),
           COUNT(*) FILTER (WHERE b.product_key IS NOT NULL),
           ROUND(100.0 * COUNT(*) FILTER (WHERE b.product_key IS NOT NULL) / COUNT(*), 2)
    FROM fact_product_nutrition f LEFT JOIN in_bridge b USING (product_key)
    """
).pl()

fact,rows,rows_with_category,coverage_pct
str,i64,i64,f64
"""weekly prices""",18699187,16923850,90.51
"""nutrition""",4334059,2721256,62.79


### 5.2 Time

Pricing and nutrition cover different periods. Their overlap bounds any analysis
that combines them; neither fact is cut to it.

In [24]:
periods = con.sql(
    """
    WITH p AS (
        SELECT MIN(d.date) AS a, MAX(d.date) AS b
        FROM fact_weekly_prices f JOIN dim_date d USING (date_key)
    ), n AS (
        SELECT MIN(d.date) AS a, MAX(d.date) AS b
        FROM fact_product_nutrition f JOIN dim_date d USING (date_key)
    )
    SELECT p.a AS pricing_start, p.b AS pricing_end,
           n.a AS nutrition_start, n.b AS nutrition_end,
           GREATEST(p.a, n.a) AS shared_start, LEAST(p.b, n.b) AS shared_end
    FROM p CROSS JOIN n
    """
).pl()
periods

pricing_start,pricing_end,nutrition_start,nutrition_end,shared_start,shared_end
date,date,date,date,date,date
2019-01-07,2020-12-28,2020-11-27,2021-02-24,2020-11-27,2020-12-28


### 5.3 Inferred products and country

Every price identity without a reference has an inferred member with country `be`,
taken from the Belgian pricing process, not from the retailer. The reserved Unknown
member is unused. Names, brands and categories of inferred members are not invented.

In [25]:
con.sql(
    """
    SELECT COUNT(*) AS rows,
           COUNT(*) FILTER (WHERE p.country IS NULL) AS rows_without_country,
           COUNT(*) FILTER (WHERE f.product_key = -1) AS unknown_product_rows,
           COUNT(*) FILTER (WHERE p.is_inferred_product) AS inferred_product_rows,
           COUNT(*) FILTER (WHERE f.product_key <> -1 AND p.country IS NULL)
               AS identified_without_country,
           COUNT(DISTINCT f.product_key) AS product_members_seen,
           COUNT(DISTINCT (f.shop_key, f.source_daltix_id)) AS source_identities_seen
    FROM fact_weekly_prices f JOIN dim_product p USING (product_key)
    """
).pl()

rows,rows_without_country,unknown_product_rows,inferred_product_rows,identified_without_country,product_members_seen,source_identities_seen
i64,i64,i64,i64,i64,i64,i64
18699187,0,0,476695,0,102069,102069


## 6. Conclusion

**Established.** Two facts at explicit grains share Product, Shop and Date; prices
add Location and nutrition adds Nutrient. The production contract passes on the
objects built above.

**Kept visible instead of hidden.** Weeks with several prices keep their range and a
flag rather than a chosen value. Unmatched price identities become inferred members that
say what they are. Retailer categories stay overlapping evidence next to one FMCG
category per product. Everything outside Belgium stays in Silver and is measured above.

**Not claimed.** No sales, volumes, margins or promotion uplift follow from price
observations alone. Nutrition dates are observation dates, and location coverage is
far smaller than the retailer networks.